# CPB Data to APEX Weather

- CBP gave us baseline hydrologic forcing data for the 4 regions
- One CSV file for each variable in each region at hourly timestep:
    - DPT: dewpoint (degC i think) --> can be used to calculate APEX relative humidity (?)
    - PRC: precipitation (in)
    - RAD: radiation (lagley/h)
    - TMP: temperature (degC) --> can find relative min/max for APEX daily TMIN and TMAX
    - WND: wind (mph)

In [3]:
# Import modules
import os
import pandas as pd
import glob

## Convert CBP data to dataframes for each region

In [ ]:
data_fp = # path to baseline hydroforcing data

In [36]:
def combine_region_data(folder_path):
    """
    Combines CSV files by region into separate DataFrames, merging variables 
    based on date columns (year, month, day, hour).
    
    Parameters:
    folder_path (str): Path to the folder containing CSV files
    
    Returns:
    dict: Dictionary with region names as keys and combined DataFrames as values
    """
    # get all CSV files in the folder
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    
    # group files by region
    region_files = {}
    for file in csv_files:
        filename = os.path.basename(file)
        region = filename.split('_')[0]
        
        if region not in region_files:
            region_files[region] = []
        
        region_files[region].append(file)
    
    # process each region's files
    region_dfs = {}
    for region, files in region_files.items():
        # use first file as base for df
        base_df = pd.read_csv(files[0])
        
        # date columns for merging
        date_columns = ['year', 'month', 'day', 'hour']
        
        # merge other variable files
        for file in files[1:]:
            df = pd.read_csv(file)
            
            # merge on date columns
            base_df = pd.merge(base_df, df, on=date_columns, how='outer')
        
        region_dfs[region] = base_df
    
    return region_dfs


In [37]:
region_dfs = combine_region_data(data_fp)


In [38]:
region_mapping = {
        'N36017': 'appalachia',  # chenago, NY
        'N51001': 'coastal',     # accomack, VA
        'N24013': 'piedmont',    # carroll, MD
        'N51820': 'valley',      # waynesboro, VA
    }


## Perform unit conversions on columns

### Define conversions as functions

Solar Radiation

In [39]:
# langley to MJ/m2

def langley_to_MJ_per_m2(langley):
    """
    Convert Langley to MJ/m2.
    
    Parameters:
    langley (float): Value in Langley
    
    Returns:
    float: Value in MJ/m2
    """
    return langley * 0.041868 #  # 1 Langley = 41868 kJ/m2 = 41868000 J/m2 = 41.868 MJ/m2 via USDA

Precipitation

In [40]:
# in to mm
def inches_to_mm(inches):
    """
    Convert inches to mm.
    
    Parameters:
    inches (float): Value in inches
    
    Returns:
    float: Value in mm
    """
    return inches * 25.4 # 1 inch = 25.4 mm

Windspeed

In [41]:
# mph to m/s
def mph_to_mps(mph):
    """
    Convert mph to m/s.
    
    Parameters:
    mph (float): Value in mph
    
    Returns:
    float: Value in m/s
    """
    return mph * 0.44704 # 1 mph = 0.44704 m/s

Relative Humidity

In [ ]:
# dewpoint to relative humidity (Magnus-Tetens equation)
def dewpoint_to_rh(temp, dewpoint):
    """
    Convert dewpoint to relative humidity using the Magnus-Tetens equation.

    Parameters:
    temp (float): Air temperature in Celsius
    dewpoint (float): Dewpoint temperature in Celsius

    Returns:
    float: Relative humidity as a fraction (0-1)
    """
    import numpy as np

    # actual vapor pressure (saturation vapor pressure evaluated at dewpoint)
    e = 611.2 * np.exp((17.62 * dewpoint) / (243.12 + dewpoint))

    # saturation vapor pressure at air temperature
    es = 611.2 * np.exp((17.62 * temp) / (243.12 + temp))

    # relative humidity as a fraction
    rh = e / es

    return rh

### perform conversions on df

In [43]:
for region, df in region_dfs.items():
    # perform unit conversions
    df['srad_mJm2'] = df['radiation(lagley/hour)'].apply(langley_to_MJ_per_m2)
    df['precip_mm'] = df['precip(in)'].apply(inches_to_mm)
    df['wind_m'] = df['windspeed(mph)'].apply(mph_to_mps)
    df['rh'] = dewpoint_to_rh(df['temperature(dC)'], df['dewpoint(°C)'])

## Format to DLY

In [ ]:
def cbp_2_dly(df, region, output_fp):
    """
    Convert CBP data to APEX .DLY format.
    
    Parameters:
    df (dataframe): dataframe for a region
    region (str): region name
    output_fp (str): path to output folder
    
    Returns:
    file: .dly file with daily data
    """
    
    # perform unit conversions
    df['srad_mJm2'] = df['radiation(lagley/hour)'].apply(langley_to_MJ_per_m2)
    df['precip_mm'] = df['precip(in)'].apply(inches_to_mm)
    df['wind_m'] = df['windspeed(mph)'].apply(mph_to_mps)
    df['rh'] = dewpoint_to_rh(df['temperature(dC)'], df['dewpoint(°C)'])

    # select only the columns we want
    region_df = df[['year', 'month', 'day', 'hour', 'srad_mJm2', 'precip_mm', 'wind_m', 'rh', 'temperature(dC)']]

    # get daily min and max temps
    t_min_max = region_df.groupby(['year', 'month', 'day'])['temperature(dC)'].agg(['min', 'max'])
    t_min_max.columns = ['tmin', 'tmax']
    t_min_max = t_min_max.reset_index()
    # merge back to original dataframe
    region_df = pd.merge(region_df, t_min_max, on=['year', 'month', 'day'])

    # group by day
    region_dly = region_df.groupby(['year', 'month', 'day']).agg({
        'srad_mJm2': 'sum',
        'precip_mm': 'sum',
        'wind_m': 'mean',
        'rh': 'mean',
        'tmin': 'min',
        'tmax': 'max'
    }).reset_index()
    
    # write to file
    output_file = os.path.join(output_fp, f"{region}.dly")

    with open(output_file, 'w') as f:
        for _, row in region_dly.iterrows():
            f.write(f'{int(row['year']):6d} {int(row['month']):3d}'
                    f'{int(row['day']):4d} {float(row['srad_mJm2']):5.1f}'
                    f'{float(row['tmax']):6.1f} {float(row['tmin']):5.1f}'
                    f'{float(row['precip_mm']):6.1f} {float(row['rh']):5.3f}'
                    f'{float(row['wind_m']):6.1f}\n')

    return region_dly

In [82]:
output_fp = r'C:\Mac\Home\Documents\APEX_scripts'

for region_code, df in region_dfs.items():
    region_name = region_mapping.get(region_code, region_code)
    cbp_2_dly(df, region_name, output_fp)

## Format to HLY

In [49]:
def cbp_2_hly(df, region, output_fp):
    """
    Convert CBP data to APEX .HLY format.
    
    Parameters:
    df (dataframe): dataframe for a region
    region (str): region name
    output_fp (str): path to output folder
    
    Returns:
    file: .hly file with hourly data
    """
    
    # perform unit conversions
    df['precip_mm'] = df['precip(in)'].apply(inches_to_mm)

    # make cumulative precip column (but only for each day)
    df['precip_cumul'] = df.groupby(['year','month','day'])['precip_mm'].cumsum()

    # select only the columns we want
    region_hly = df[['year', 'month', 'day', 'hour', 'precip_cumul']]
    
    # write to file
    output_file = os.path.join(output_fp, f"{region}.hly")

    with open(output_file, 'w') as f:
        for _, row in region_hly.iterrows():
            f.write(f'{int(row['year']):4d} {int(row['month']):3d} {int(row['day']):3d} {int(row['hour']):9d} {float(row['precip_cumul']):9.3f}\n')
            
    return region_hly

In [50]:
output_fp = r'C:\Mac\Home\Documents\APEX_scripts'

for region_code, df in region_dfs.items():
    region_name = region_mapping.get(region_code, region_code)
    cbp_2_hly(df, region_name, output_fp)

## Wind

### Calculate monthly wind in m/s

In [ ]:
## assume at 10m height
def cbp_2_wnd(df, region, output_fp):
    """
    Convert CBP data to APEX .WND format.
    
    Parameters:
    df (dataframe): dataframe for a region
    region (str): region name
    output_fp (str): path to output folder
    
    Returns:
    file: .wnd file with monthly wind data
    """
    
    # perform unit conversions
    df['wind_m'] = df['windspeed(mph)'].apply(mph_to_mps)

    # get monthly mean wind speed
    monthly_wind = df.groupby(['month'])['wind_m'].agg('mean')
    monthly_wind_list = monthly_wind.tolist()

    # write to file
    output_file = os.path.join(output_fp, f"{region}.wnd")

    with open(output_file, 'w') as f:
        # write header
        f.write(f"{region}  \n")
        f.write("from Chesapeake Bay Program (10m)\n")
        
        wind_speed_str = "  ".join([f"{wind:.2f}" for wind in monthly_wind_list]) + "  WSPD  \n"
        f.write(wind_speed_str)
    
    return output_file

In [72]:
output_fp = r'C:\Users\mayastruzak\Box\CBW\data\apex_hydro_base'

for region_code, df in region_dfs.items():
    region_name = region_mapping.get(region_code, region_code)
    cbp_2_wnd(df, region_name, output_fp)

# Mean Atmospheric Deposition

In [11]:
def acre2ha(acre):
    """
    Convert acres to hectares.
    
    Parameters:
    acre (float): Value in acres
    
    Returns:
    float: Value in hectares
    """
    return acre * 0.404686 # 1 acre = 0.404686 ha


In [ ]:
import glob
import pandas as pd
import os

atm_data_fp = # path to atm deposition data
# county_area = {
#     'N10001': acre2ha(131206),
#     'N24013': acre2ha(289718), 
#     'N24039': acre2ha(142378),
#     'N24043': acre2ha(299096),
#     'N36017': acre2ha(131206), # chenago, NY
#     'N42021': acre2ha(187853),
#     'N42087': acre2ha(265416),
#     'N51001': 0, # accomack, VA no area data
#     'N51145': acre2ha(167901), 
#     'N51820': 0 # no area data
# }

# CBP conversion factor lb/ac/in to mg/l
conversion = 4.41279983


# get all CSV files in the folder
csv_files = glob.glob(os.path.join(atm_data_fp, "*.csv"))

site_data = {}
for file in csv_files:
    # get site id from filename
    filename = os.path.basename(file)
    site_id = filename.split('_')[1].split('.')[0]

    # read csv file into dataframe
    site_df = pd.read_csv(file, delimiter=',')

    # sum total N deposited
    wetN = (site_df['wetno3(lb)'] + site_df['wetnh3(lb)'] + site_df['wetorn(lb)']).sum()
    
    # sum total precip
    total_precip_in = (site_df['precip(in)']).sum() 
    
    wetN_mgL = conversion * wetN / total_precip_in

    
    site_data[site_id] = wetN_mgL

    # write to new dataframe
atm_dep_site_df = pd.DataFrame(site_data.items(), columns=['site_id','avg_N_mg/L'])
    
atm_dep_site_df.to_csv(os.path.join(atm_data_fp, 'avg_atmd.csv'), index=False)

In [65]:
merged_df = pd.merge(atm_dep_site_df, region_keys_df, on='site_id', how='left')
# merged_df.to_csv(os.path.join(atm_data_fp, 'avg_atmd_region.csv'), index=False)
merged_df 

,site_id,avg_N_mg/L,county,state,region
0,N42087,0.000003,Mifflin,PA,valley and ridge
1,N51001,0.000000,Accomack,VA,coastal
2,N51820,0.000000,Waynesboro,VA,valley and ridge
3,N36017,0.000006,Chenango,NY,appalachia
4,N24013,0.000002,Carroll,MD,piedmont
5,N10001,0.000004,Kent,DE,coastal
6,N42021,0.000005,Cambria,PA,appalachia
7,N24039,0.000003,Somerset,MD,coastal
8,N24043,0.000003,Washington,MD,coastal
9,N51145,0.000003,Powhatan,VA,piedmont


In [130]:
# get regional average
region_avg_df = merged_df.groupby('region')['avg_N_mg/L'].mean().reset_index()
region_avg_df.to_csv(os.path.join(atm_data_fp, 'atmd_regional_avg.csv'), index=False)
region_avg_df

,region,avg_N_mg/L
0,appalachia,0.064577
1,coastal,0.044807
2,piedmont,0.043837
3,valley and ridge,0.051277
